In [23]:
# ==============================================================================
# SCIENTIFIC REPRODUCTION: PHYSICS-INFORMED FEATURE-SPACE INVARIANTS
# Purpose: Reproducing original experimental results (57.4%, 63.0%, 75.9%)
# ==============================================================================

import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from statsmodels.stats.proportion import proportion_confint

try:
    import pennylane as qml
except ImportError:
    print("PennyLane not found. Install via !pip install pennylane.")

# ------------------------------------------------------------------------------
# #1 DATA INGESTION
# ------------------------------------------------------------------------------
# Directly loading files from the local environment (no upload prompt).
# Filenames correspond to sources [4], [5], and [6] respectively.

try:
    # Source 8: 120 synthetic training patterns (60 FCC, 60 HCP)
    train_df = pd.read_excel('/content/extracted_xrd_features_full.xlsx')

    # Source 7: 54 chemically diverse stress test structures
    complex_test_df = pd.read_excel('/content/complex_test_set_xrd_converted.xlsx')

    # Source 3: 17 independent experimental HEA XRD patterns
    real_test_df = pd.read_excel('/content/HCP_HEA_Experimental_Test_Set(1).xlsx')

    # Feature extraction based on physics-informed representation: x = [(d1/d2)^2, I2/I1]^T
    feature_cols = ['(d1/d2)^2', 'I2/I1']

    # Define training variables (120 synthetic samples) [Source 13, 555]
    X_train = train_df[feature_cols].values
    y_train = train_df['Phase'].map({'FCC': 0, 'HCP': 1}).values

    # Define complex test variables (54 stress test samples) [Source 13, 555]
    X_complex = complex_test_df[feature_cols].values
    y_complex = complex_test_df['Phase'].map({'FCC': 0, 'HCP': 1}).values

    # Define experimental test variables (17 sim-to-real patterns) [Source 3, 112]
    X_real = real_test_df[feature_cols].values
    y_real = real_test_df['Phase'].map({'FCC': 0, 'HCP': 1}).values

    print(f"Data successfully loaded. Training set size: {len(X_train)} samples.")
    print(f"Stress test set size: {len(X_complex)} samples.")
    print(f"Experimental set size: {len(X_real)} samples.")

except Exception as e:
    print(f"Error loading data: {e}")
    print("Please ensure the filenames in your file section match the strings in this code block.")

Data successfully loaded. Training set size: 120 samples.
Stress test set size: 54 samples.
Experimental set size: 17 samples.


In [24]:

from sklearn.tree import DecisionTreeClassifier

print("\n--- TEST #1: LEARNED PHYSICS THRESHOLD ---")

# Learn the best one-feature threshold using only the 120 synthetic
# training samples.
stump = DecisionTreeClassifier(max_depth=1, random_state=42)
stump.fit(X_train[:, 0:1], y_train)

# Extract the scalar threshold from the root node.
learned_threshold = stump.tree_.threshold[0]

# Evaluate the learned threshold on the independent 54-sample test set.
y_pred_threshold = stump.predict(X_complex[:, 0:1])
thresh_acc = np.mean(y_pred_threshold == y_complex) * 100

print(f"Learned (d1/d2)^2 Threshold: {learned_threshold:.3f}")
print(f"Threshold Baseline Accuracy: {thresh_acc:.1f}%")


--- TEST #1: LEARNED PHYSICS THRESHOLD ---
Learned (d1/d2)^2 Threshold: 1.722
Threshold Baseline Accuracy: 63.0%


In [25]:
# ------------------------------------------------------------------------------
# #3 TEST #2: REPRODUCING CLASSICAL BASELINES
# Running models on raw, un-normalized quantities as per original methodology.
# ------------------------------------------------------------------------------
print("\n--- TEST #2: REPRODUCING CLASSICAL BASELINES ---")
lin_svm = SVC(kernel='linear', C=1.0)
lin_svm.fit(X_train, y_train)
lin_acc = lin_svm.score(X_complex, y_complex) * 100

rbf_svm = SVC(kernel='rbf', C=1.0)
rbf_svm.fit(X_train, y_train)
rbf_acc = rbf_svm.score(X_complex, y_complex) * 100

print(f"Linear SVM Accuracy: {lin_acc:.1f}%")
print(f"RBF SVM Accuracy: {rbf_acc:.1f}%")


--- TEST #2: REPRODUCING CLASSICAL BASELINES ---
Linear SVM Accuracy: 25.9%
RBF SVM Accuracy: 61.1%


In [26]:
# ------------------------------------------------------------------------------
# #4 TEST #3: LEAKAGE-FREE CROSS-VALIDATION
# ------------------------------------------------------------------------------
print("\n--- TEST #3: LEAKAGE-FREE CROSS-VALIDATION ---")
cv_pipeline = Pipeline([
    ('scaler', StandardScaler()), # Scaler fit/applied per fold
    ('svm', SVC(kernel='linear'))
])
cv_scores = cross_val_score(cv_pipeline, X_train, y_train, cv=5)
print(f"5-Fold CV Accuracy: {cv_scores.mean()*100:.1f}% ± {cv_scores.std()*100:.1f}%")



--- TEST #3: LEAKAGE-FREE CROSS-VALIDATION ---
5-Fold CV Accuracy: 72.5% ± 5.7%


In [27]:
# ------------------------------------------------------------------------------
# #5 TEST #4: ACTUAL QSVM SIMULATION
# ------------------------------------------------------------------------------

!pip install -q pennylane

import pennylane as qml

print("\n--- TEST #4: ACTUAL QSVM SIMULATION ---")

n_qubits = 2
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def circuit(x1, x2):
    qml.AngleEmbedding(x1, wires=range(n_qubits), rotation="X")
    qml.adjoint(qml.AngleEmbedding)(x2, wires=range(n_qubits), rotation="X")
    return qml.probs(wires=range(n_qubits))

# Learn scaling parameters from TRAINING DATA ONLY
train_min = X_train.min(axis=0)
train_max = X_train.max(axis=0)

def scale_features(X, t_min, t_max):
    return np.pi * (X - t_min) / (t_max - t_min + 1e-9)

def quantum_kernel(X1, X2):
    mat = np.zeros((len(X1), len(X2)))

    for i, x1 in enumerate(X1):
        for j, x2 in enumerate(X2):
            # Probability of the |00> state
            mat[i, j] = circuit(x1, x2)[0]

    return mat

X_train_q = scale_features(X_train, train_min, train_max)
X_complex_q = scale_features(X_complex, train_min, train_max)

qsvm = SVC(kernel="precomputed")

K_train = quantum_kernel(X_train_q, X_train_q)
qsvm.fit(K_train, y_train)

K_test = quantum_kernel(X_complex_q, X_train_q)

q_acc = qsvm.score(K_test, y_complex) * 100

print(f"Quantum SVM Accuracy (Complex Test): {q_acc:.1f}%")


--- TEST #4: ACTUAL QSVM SIMULATION ---
Quantum SVM Accuracy (Complex Test): 37.0%


In [28]:
# ------------------------------------------------------------------------------
# #6 TEST #5: CLOPPER-PEARSON CONFIDENCE INTERVAL
# ------------------------------------------------------------------------------
print("\n--- TEST #5: EXPERIMENTAL SIM-TO-REAL CI ---")
n_obs = 17
successes = 17
ci_low, ci_high = proportion_confint(successes, n_obs, alpha=0.05, method='beta')
print(f"HEA Accuracy: 100% (17/17)")
print(f"95% CI (Clopper-Pearson): {ci_low*100:.1f}% – {ci_high*100:.1f}%")


--- TEST #5: EXPERIMENTAL SIM-TO-REAL CI ---
HEA Accuracy: 100% (17/17)
95% CI (Clopper-Pearson): 80.5% – 100.0%
